# Time-Resolved XRD Analysis

Load compact processed 1-D stacks into xarray, keep raw/cake rows lazy,
normalize and bin with provenance, inspect a pilot fit, then derive a
lattice and explicitly illustrative temperature history. Physical rates
require explicit seconds and a calibration.


In [ ]:
import tempfile
from pathlib import Path

import h5py
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.analysis import (
    LinearThermalExpansion, PeakFitPlan, add_lattice_results,
    add_temperature_results, bin_time_resolved, fit_peak_series,
    export_time_resolved_results,
    flag_fit_quality, flag_normalization_outliers, load_time_resolved_series,
    normalize_monitor, normalize_reference_band, select_time_zero,
)
from xrd_tools.gui.widgets import ImageViewer, PatternViewer, PeakFitControls
from xrd_tools.viz import plot_peak_fit_frame, plot_thermal_history, plot_time_resolved_waterfall


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
processed_root = TEST_DATA / "xdart_processed_data"
processed_file = widgets.Text(value=str(processed_root / "Pt_test_burst_00007.nxs"), description="processed file")
processed_folder = widgets.Text(value=str(processed_root), description="processed folder")
selection_mode = widgets.ToggleButtons(options=("file", "folder"), value="file", description="load")
raw_root = widgets.Text(value=str(TEST_DATA) if TEST_DATA.exists() else "", description="raw root")
time_key = widgets.Text(value="", description="time key")
time_unit = widgets.Dropdown(options=("s", "ms", "us"), value="s", description="time unit")
frame_period_ms = widgets.BoundedFloatText(value=2.0, min=0.001, max=1e6, step=0.1, description="period ms")
monitor_method = widgets.Dropdown(options=("reference band", "monitor"), value="reference band", description="normalize")
monitor_key = widgets.Text(value="i0", description="monitor key")
q_band = widgets.FloatRangeSlider(value=(3.05, 3.20), min=2.4, max=3.3, step=0.01, description="reference q", continuous_update=False)
bin_size = widgets.BoundedIntText(value=2, min=1, max=100, description="bin size")
frame_row = widgets.BoundedIntText(value=0, min=0, max=0, description="row")
batch_limit = widgets.BoundedIntText(value=8, min=1, max=1000, description="batch rows")
load_button = widgets.Button(description="Load processed", button_style="primary")
preprocess_button = widgets.Button(description="Apply preprocessing")
inspect_button = widgets.Button(description="Inspect lazy row")
pilot_button = widgets.Button(description="Run pilot fit", button_style="primary")
batch_button = widgets.Button(description="Run bounded batch", button_style="primary")
export_button = widgets.Button(description="Export compact results")
export_directory = Path(os.environ.get("XDART_NOTEBOOK_OUTPUT", tempfile.gettempdir()))
status = widgets.HTML("<i>Load, preprocess, and fit actions are explicit; row selection only reads cached/lazy data.</i>")
output = widgets.Output()
display(widgets.VBox([selection_mode, processed_file, processed_folder, raw_root, widgets.HBox([time_key, time_unit, frame_period_ms]), widgets.HBox([monitor_method, monitor_key]), q_band, widgets.HBox([bin_size, frame_row, batch_limit]), widgets.HBox([load_button, preprocess_button, inspect_button]), widgets.HBox([pilot_button, batch_button, export_button]), status, output]))


In [ ]:
NOTEBOOK_STATE = {"loads": 0, "preprocesses": 0, "series": None, "prepared": None, "fits": None, "thermal": None, "raw_reads": 0}

def _smoke_file():
    path = Path(tempfile.gettempdir()) / "xdart_notebook_time_resolved_smoke.nxs"
    q, frames = np.linspace(2.45, 3.30, 240), np.arange(16, dtype=np.int64)
    centers = 2.765 - 0.0008 * frames
    stack = np.array([15 + 180 * np.exp(-0.5 * ((q - center) / 0.018) ** 2) for center in centers], dtype=np.float32)
    with h5py.File(path, "w") as h5:
        entry = h5.create_group("entry")
        one_d = entry.create_group("integrated_1d"); one_d.create_dataset("frame_index", data=frames)
        q_data = one_d.create_dataset("q", data=q); q_data.attrs["units"] = "q_A^-1"
        one_d.create_dataset("intensity", data=stack); one_d.create_dataset("sigma", data=np.sqrt(stack))
        scan_data = entry.create_group("scan_data"); scan_data.create_dataset("frame_index", data=frames)
        scan_data.create_dataset("elapsed", data=frames * 2.0); scan_data.create_dataset("i0", data=np.linspace(0.98, 1.02, len(frames)))
        cakes = entry.create_group("integrated_2d"); cakes.create_dataset("frame_index", data=frames); cakes.create_dataset("q", data=q); cakes.create_dataset("chi", data=np.linspace(-30, 30, 12)); cakes.create_dataset("intensity", data=np.broadcast_to(stack[:, None, :], (len(frames), 12, len(q))))
        groups = entry.create_group("frames")
        for frame in frames:
            groups.create_group(f"frame_{frame:04d}").create_dataset("thumbnail", data=np.ones((8, 8), dtype=np.uint8))
    return path

def _selected_paths():
    if SMOKE_MODE:
        return _smoke_file()
    candidate = Path(processed_file.value).expanduser() if selection_mode.value == "file" else Path(processed_folder.value).expanduser()
    assert candidate.exists(), f"Missing processed selection: {candidate}"
    return candidate

def load_processed(_=None):
    with output:
        clear_output(wait=True)
        try:
            selected_key = "elapsed" if SMOKE_MODE else time_key.value.strip() or None
            selected_unit = "ms" if SMOKE_MODE else (time_unit.value if selected_key else None)
            series = load_time_resolved_series(_selected_paths(), frame_period_s=frame_period_ms.value / 1000.0, time_key=selected_key, time_unit=selected_unit, metadata_keys=(monitor_key.value.strip(),) if monitor_key.value.strip() else (), source_root=raw_root.value.strip() or None)
            frame_row.max, frame_row.value = series.dataset.sizes["pattern"] - 1, 0
            NOTEBOOK_STATE.update(loads=NOTEBOOK_STATE["loads"] + 1, series=series, prepared=None, fits=None, thermal=None)
            status.value = f"<b>Loaded {series.dataset.sizes['pattern']} scan-qualified patterns.</b>"
            display({"patterns": series.dataset.sizes["pattern"], "q_points": series.dataset.sizes["q"], "time_units": series.dataset.coords["time"].attrs["units"]})
        except Exception as exc:
            status.value = f"<b>Load failed:</b> {exc}"
            raise

def apply_preprocessing(_=None):
    with output:
        clear_output(wait=True)
        try:
            series = NOTEBOOK_STATE["series"]
            assert series is not None, "Load processed data first"
            dataset, fallback = series.dataset, False
            if monitor_method.value == "monitor":
                try:
                    candidate = normalize_monitor(dataset, monitor_key.value.strip())
                    if bool(candidate["monitor_normalization_valid"].any()):
                        dataset, source_var = candidate, "intensity_normalized"
                    else:
                        fallback, source_var = True, "intensity"
                except (KeyError, ValueError):
                    fallback, source_var = True, "intensity"
            else:
                source_var = "intensity"
            normalized = normalize_reference_band(dataset, q_range=tuple(q_band.value), intensity_var=source_var, output_var="intensity_band_normalized")
            prepared = select_time_zero(bin_time_resolved(flag_normalization_outliers(normalized), bin_size=bin_size.value), zero_pattern=0)
            NOTEBOOK_STATE.update(preprocesses=NOTEBOOK_STATE["preprocesses"] + 1, prepared=prepared, fits=None, thermal=None)
            display(plot_time_resolved_waterfall(prepared, intensity_var="intensity_band_normalized", log_intensity=True))
            status.value = "<b>Reference-band preprocessing complete.</b>" if not fallback else "<b>Selected monitor was unavailable; used documented reference-band fallback.</b>"
        except Exception as exc:
            status.value = f"<b>Preprocessing failed:</b> {exc}"
            raise

def inspect_lazy_row(_=None):
    with output:
        clear_output(wait=True)
        try:
            series = NOTEBOOK_STATE["series"]
            assert series is not None, "Load processed data first"
            row = int(frame_row.value)
            thumbnail, cake = series.get_thumbnail(row), series.get_cake(row)
            try:
                image, title, raw_available = series.get_raw(row), "Full raw detector frame", True
            except (KeyError, FileNotFoundError, ValueError):
                image, title, raw_available = thumbnail, "Stored thumbnail (raw unavailable)", False
            pattern = series.dataset.isel(pattern=row)
            display(widgets.HBox([PatternViewer(patterns=[(pattern.q.values, pattern.intensity.values, f"row {row}")]).widget, ImageViewer(image, title=title).widget]))
            display({"scan": str(pattern.scan_name.item()), "frame_label": int(pattern.frame_label.item()), "cake_shape": cake.intensity.shape, "raw_available": raw_available})
            NOTEBOOK_STATE["raw_reads"] += 1
        except Exception as exc:
            status.value = f"<b>Lazy inspection failed:</b> {exc}"
            raise

def _plan_from_controls(params):
    positions = tuple(params["positions"] or (2.76,))
    return PeakFitPlan(positions=positions, model=params["model"], background=params["background"] if params["background"] in {"none", "constant", "linear"} else "linear", sigma_init=params["sigma_init"], sigma_bounds=params["sigma_bounds"], center_bounds_delta=params["center_bounds_delta"], fit_kwargs={"method": "leastsq"})

def run_pilot(params=None):
    with output:
        clear_output(wait=True)
        try:
            prepared = NOTEBOOK_STATE["prepared"]
            assert prepared is not None, "Apply preprocessing first"
            fits = fit_peak_series(prepared, _plan_from_controls(params or fit_controls.get_params()), intensity_var="intensity_band_normalized", q_range=(fit_controls.q_min.value, fit_controls.q_max.value), pattern_indices=[min(int(frame_row.value), prepared.sizes["pattern"] - 1)])
            NOTEBOOK_STATE["fits"] = flag_fit_quality(fits, max_center_error=0.02)
            display(plot_peak_fit_frame(NOTEBOOK_STATE["fits"], 0))
            status.value = "<b>Pilot fit complete.</b>"
        except Exception as exc:
            status.value = f"<b>Pilot fit failed:</b> {exc}"
            raise

fit_controls = PeakFitControls(on_fit=run_pilot)
peak_center = 2.76 if SMOKE_MODE else 1.56
fit_controls.n_peaks.value = 1; fit_controls.peak_positions.value = str(peak_center); fit_controls.peak_model.value = "gaussian"; fit_controls.bg_model.value = "linear"; fit_controls.sigma_init.value = 0.02; fit_controls.q_min.value, fit_controls.q_max.value = peak_center - 0.16, peak_center + 0.16
display(fit_controls.widget)

def run_bounded_batch(_=None):
    with output:
        clear_output(wait=True)
        try:
            prepared = NOTEBOOK_STATE["prepared"]
            assert prepared is not None, "Apply preprocessing first"
            fits = flag_fit_quality(fit_peak_series(prepared, _plan_from_controls(fit_controls.get_params()), intensity_var="intensity_band_normalized", q_range=(fit_controls.q_min.value, fit_controls.q_max.value), pattern_indices=np.arange(min(batch_limit.value, prepared.sizes["pattern"]))), max_center_error=0.02)
            lattice = add_lattice_results(fits, hkls=((1, 1, 1),))
            calibration = LinearThermalExpansion(float(lattice.lattice_mean_A.isel(fit_pattern=0)), 300.0, 9e-6)
            thermal = add_temperature_results(lattice, calibration, time_coord="time")
            NOTEBOOK_STATE.update(fits=fits, thermal=thermal)
            display(plot_thermal_history(thermal))
            status.value = f"<b>Saved {thermal.sizes['fit_pattern']} bounded fit rows for review.</b>"
        except Exception as exc:
            status.value = f"<b>Batch fit failed:</b> {exc}"
            raise

def export_compact(_=None):
    thermal = NOTEBOOK_STATE["thermal"]
    assert thermal is not None, "Run bounded batch before exporting"
    paths = export_time_resolved_results(thermal, netcdf_path=export_directory / "time_resolved_results.nc", csv_path=export_directory / "time_resolved_scalars.csv")
    status.value = f"<b>Exported compact results to {paths['netcdf'].parent}.</b>"
    return paths

load_button.on_click(load_processed); preprocess_button.on_click(apply_preprocessing); inspect_button.on_click(inspect_lazy_row); pilot_button.on_click(run_pilot); batch_button.on_click(run_bounded_batch); export_button.on_click(export_compact)
frame_row.observe(lambda change: inspect_lazy_row() if NOTEBOOK_STATE["series"] is not None else None, names="value")
NOTEBOOK_ACTIONS = {"load_processed": load_processed, "apply_preprocessing": apply_preprocessing, "inspect_lazy_row": inspect_lazy_row, "run_pilot": run_pilot, "run_bounded_batch": run_bounded_batch, "export_compact": export_compact}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    load_processed(); apply_preprocessing(); inspect_lazy_row(); run_pilot(); run_bounded_batch()
